# BALoRA final full training and research run

**Raw BTCV → fresh training → validation selection → full test evaluation → figures and checkpoints.**

This is the full **Fold 1, one-seed** research pipeline. It trains both deterministic rank-16 LoRA and BALoRA from the official pretrained backbone. Frozen SAM2 is evaluated separately. Previous trained-results archives and saved scores are **not required**. All reported new metrics, epochs, curves and masks come from the current run; its results are not assumed to match the previous experiment.

The task is **2D prompt-conditioned binary organ segmentation with a positive point sampled from the ground-truth mask**. It retains all positive slices, 1024 × 1024 inputs, 19/5/6 volumes and BALoRA Ktrain=2/Kinfer=4. Deterministic methods make one prediction and have no artificial epistemic map. The primary no-refinement policy is fixed before this new fit.

Enable **GPU T4 x2 and Internet**, attach the raw BTCV NIfTIs, review the configuration and use **Save Version → Save & Run All**. Separate model processes can train on both GPUs. Final tests run sequentially on one GPU for matched timing boundaries.


## Labeled BTCV root repair v4.2

The confirmed failure came from scanning extra img0061–img0080 images without labels. The notebook now finds a verified complete labeled Training folder before SAM2 setup and applies that root to every stage. All 30 study volumes and the fixed 19/5/6 split remain required. Missing study masks and ambiguous dataset copies still fail explicitly. Attach results ZIPs from all failed attempts to preserve their ledgers.


## Time budget and resume

The existing experimental ledger already contains **20.8847435729 hours**. Keeping the original 30-hour cumulative cap leaves about **9.1153 hard hours**, before the safety reserve. A complete fresh rerun may require more time. The code carries the old usage forward, profiles the real workloads, actually runs the training stages while allowed, and saves a **PARTIAL** state if a deadline arrives. It never shortens the protocol or calls partial work complete.

Kaggle documents a 12-hour CPU/GPU session limit. This notebook requests at most 10.5 work hours per session and reserves export time. If the run spans sessions, attach its previous results ZIP and run the same notebook again. It automatically finds one matching v4 study and resumes full training/evaluation state. An expanded compute allowance must be explicitly entered in `total_cumulative_hours`; the original cap and all previous usage remain recorded.


In [ ]:
import os, time
NOTEBOOK_START = time.time()
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

CONFIG = {
    'study_id': 'balora_full_run_fold1_seed42_v4',
    'seed': 42,
    'input_root': os.environ.get('BALORA_INPUT_ROOT', '/kaggle/input'),
    'btcv_root': '',                    # optional directory containing only the intended 30 BTCV cases
    'resume_root': '',                  # auto-detect one matching v4 output; or set ZIP/folder explicitly
    'session_work_hours': 10.5,         # notebook session cap, plus export margin
    'total_cumulative_hours': 30.0,     # retains 20.8847435729 historical hours; does NOT reset them
    'parallel_on_two_gpus': True,
    'reconciliations': {},
}
# A fresh repeat is additional experimental work. Only if you explicitly
# allocate more total compute, increase total_cumulative_hours before running.
# All old usage and the original 30-hour cap remain visible in the ledger.
# For a killed prior session, reconcile its exact session ID using the FULL
# elapsed host duration and evidence, never the last heartbeat alone:
# CONFIG['reconciliations'] = {
#     'exact_session_id': {'full_elapsed_seconds': 43213.5,
#                          'host_log_evidence': 'Kaggle version X full elapsed-time log'}
# }


## Extract and verify the full executable source

The embedded payload contains model/training/evaluation source, configuration, figure code and tests. It contains no trained adapter checkpoint, historical test-score table or patient image. The runtime verifies the official SAM2 commit, backbone hash and source provenance. Non-core helpers are installed only when missing; CUDA-enabled PyTorch is retained.


In [ ]:
import base64, hashlib, io, zipfile
from pathlib import Path
PAYLOAD_B64 = '...'
EXPECTED_PAYLOAD_SHA256 = '5e58ad385d81651cceeba6e8b5ccd1e4faea009cee770a9780d43df4518a7d6a'
raw = base64.b64decode(PAYLOAD_B64)
if hashlib.sha256(raw).hexdigest() != EXPECTED_PAYLOAD_SHA256:
    raise ValueError('Embedded full-run source checksum mismatch')
BASE = Path(os.environ.get('BALORA_WORKSPACE_BASE', '/kaggle/working' if Path('/kaggle/working').is_dir() else str(Path.cwd())))
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE/'balora_full_run_source_v4'
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if z.testzip() is not None: raise ValueError('Damaged embedded source')
    for name in z.namelist():
        if not (PROJECT/name).resolve().is_relative_to(PROJECT.resolve()) or '\\' in name:
            raise ValueError('Unsafe archive path')
    z.extractall(PROJECT)
CONFIG['output_root'] = str(BASE/CONFIG['study_id'])
print('Verified full-run source:', PROJECT)
print('Output:', CONFIG['output_root'])


## Execute the complete pipeline

This cell runs all stages, including **fresh training**. Profiles use disposable weights; research fitting starts from fresh adapters with the declared seed. Each fitted model selects its own checkpoint on fixed-prompt validation Dice using a 20-epoch maximum, eight-epoch minimum and patience six.

After both new fits complete, the pipeline audits legacy and guarded refinement on validation only, keeps the predeclared no-refinement main comparison, evaluates every test target for all three methods, exports regional calibration and uncertainty diagnostics, and builds all figures. The new refinement findings are reported regardless of whether they repeat the historical outcome.

Checkpoints retain optimizer, scheduler, GradScaler, RNG and partial-epoch position. Saved test records have checkpoint, data, source, prompt and runtime provenance. Model namespaces run in isolated processes, and the official SAM2 builder is resolved before model construction.


In [ ]:
import sys, json
sys.path.insert(0, str(PROJECT))
import importlib, full_pipeline
full_pipeline = importlib.reload(full_pipeline)
run = full_pipeline.run
RUN_STATUS, RESULTS_ZIP = run(CONFIG, started=NOTEBOOK_START)
print(json.dumps(RUN_STATUS, indent=2))
print('Full-state output ZIP:', RESULTS_ZIP)


## Inspect completion and newly measured results

Only **FULL_NEW_FOLD1_PROTOCOL_COMPLETE** and `full_new_protocol_complete=true` establish completion of this new pipeline. The already completed historical comparison remains separate. An archive, profile, passing test or partial training curve is not a completed new experiment.


In [ ]:
OUTPUT = Path(CONFIG['output_root'])
status = json.loads((OUTPUT/'run_status.json').read_text())
print('Execution status:', status['status'])
print('New full protocol complete:', status['full_new_protocol_complete'])
report = OUTPUT/'final_research_report.json'
if report.is_file() and status['full_new_protocol_complete']:
    result = json.loads(report.read_text())
    for row in result['metrics']:
        if row['metric'] == 'dice': print(row['method'], row['mean'], 'defined pairs', row['defined_pairs'])
    print('Coverage:', result['coverage'])
    print('New paired-volume effects:', result['volume_contrasts'])
else:
    print('This new run is incomplete. Attach its results ZIP and resume; inspect the ledger before continuing.')


## All figures generated from this run

Full completion produces **13 figure sets** in PNG/SVG/PDF: schematic; actual training/validation curves; per-organ and paired-volume Dice; primary calibration; 2D surface distances; the new validation refinement audit; rank use; all paired differences; case-deletion sensitivity; risk–coverage; conditional reliability; and measured inference time/memory. It also creates genuine qualitative panels for predeclared maximum-area liver, pancreas and right-adrenal slices in each test volume.

The qualitative rule is set from anatomy before fitting, rather than favorable model scores. Panels show actual CT, target, three predictions, the same oracle point and Bayesian MI. No raw examples or new curves are fabricated when fitting/evaluation is partial. A partial run may show its explicitly labeled training history only.


In [ ]:
try:
    from IPython.display import display, Image, Markdown, FileLink
except ImportError:
    print('Figures:', OUTPUT/'figures')
else:
    captions_path = OUTPUT/'figure_captions.json'
    captions = json.loads(captions_path.read_text()) if captions_path.is_file() else {}
    for path in sorted((OUTPUT/'figures').glob('figure_*.png')):
        display(Image(filename=str(path), width=1100))
        if path.stem in captions: display(Markdown(captions[path.stem]))
    for path in sorted((OUTPUT/'figures/qualitative').glob('*.png')):
        display(Image(filename=str(path), width=1100))
    display(FileLink(RESULTS_ZIP))


## Save or resume

Download **BALoRA_Full_Run_Results.zip** or save the notebook output as **balora-full-run-fold1-seed42-v4**. For another session, attach the previous v4 ZIP as an input and rerun this same notebook. `resume_root` is needed only when discovery is ambiguous. Keep every prior follow-up ledger attached. If the cumulative allowance is exhausted, additional sessions alone do not create more budget.

Primary metrics average slices within each volume-organ pair, then pairs equally. Six test volumes contribute 460 foreground slices, 2,466 predictions and 78 dependent pairs per method. Whole volumes are the independent bootstrap units; intervals are exploratory and do not cover training-seed or external variation. HD95/ASSD are resized **2D pixels**, not 3D millimetres. All-pixel calibration is background-sensitive. Conditional regions use ground truth, histogram ranking is approximate, and deterministic entropy is not epistemic MI.

This is a complete executable **single-fold, single-seed** research recipe, not proof of an already executed new GPU experiment, five-fold validation or clinical reliability. Engineering validation is documented separately. [Kaggle technical specifications](https://www.kaggle.com/docs/notebooks) and [official SAM2 installation](https://github.com/facebookresearch/sam2/blob/main/INSTALL.md) describe the execution environment; they do not guarantee a finish time or error-free training.
